In [ ]:
from ADFWI.survey import Receiver, SeismicData, Source, Survey
from ADFWI.utils import wavelet

from scipy import integrate

ox, oz  = 0, 0             # Origin coordinates for x and z directions
nz, nx  = 76, 200          # Grid dimensions in z and x directions
dx, dz  = 40, 40           # Grid spacing in x and z directions
nt, dt  = 2500, 0.003      # Time steps and time interval
nabc    = 30               # Thickness of the absorbing boundary layer
f0      = 5                # Initial frequency in Hz
free_surface = True        # Enable free surface boundary condition

# Define source positions in the model
src_z = np.array([1 for i in range(2, nx-1, 5)])  # Z-coordinates for sources
src_x = np.array([i for i in range(2, nx-1, 5)])  # X-coordinates for sources
src_t, src_v = wavelet(nt, dt, f0, amp0=1)  # Create time and wavelet amplitude
src_v = integrate.cumtrapz(src_v, axis=-1, initial=0)  # Integrate wavelet to get velocity
source = Source(nt=nt, dt=dt, f0=f0)  # Initialize source object
# Method 2: Loop through each source position to add them individually
for i in range(len(src_x)):
    source.add_source(src_x=src_x[i], src_z=src_z[i], src_wavelet=src_v, src_type="mt", src_mt=np.array([[1,0,0],[0,1,0],[0,0,1]]))

# Define receiver positions in the model
rcv_z = np.array([1 for i in range(0, nx, 1)])  # Z-coordinates for receivers
rcv_x = np.array([j for j in range(0, nx, 1)])  # X-coordinates for receivers
receiver = Receiver(nt=nt, dt=dt)  # Initialize receiver object
# Method 2: Loop through each receiver position to add them individually
for i in range(len(rcv_x)):
    receiver.add_receiver(rcv_x=rcv_x[i], rcv_z=rcv_z[i], rcv_type="pr")

# Create a survey object using the defined source and receiver
survey = Survey(source=source, receiver=receiver)

# Create a SeismicData object to store observed data from the survey
d_obs = SeismicData(survey)

# Record the waveform data into the SeismicData object
clean_data_path = "../Inductive_bias/regularization/Marmousi2-nowater-RealNoise/RealNoiseIRIS-Smooth=6/snr=1/waveform/obs_data_clean.npz"
d_obs.load(clean_data_path)

clean_data = d_obs.data["p"].copy()

clean_data.shape,clean_data.min(),clean_data.max()

In [ ]:
noise_data_path = "../Inductive_bias/regularization/Marmousi2-nowater-RealNoise/RealNoiseGuangYuan-Smooth=6/snr=1/waveform/obs_data.npz"

d_obs.load(noise_data_path)

noisy_data = d_obs.data["p"]

noisy_data.shape

In [ ]:
plt.figure()
shot_idx = 10
plt.subplot(1,2,1)
plt.imshow(clean_data[shot_idx,:,:],cmap="seismic",aspect="auto")
plt.subplot(1,2,2)
plt.imshow(noisy_data[shot_idx,:,:],cmap="seismic",aspect="auto")
plt.show()

In [ ]:
plt.figure()
shot_idx = 10
rcv_idx = 20
plt.figure(figsize=(10,4))
plt.plot(noisy_data[shot_idx,:,rcv_idx],label="noisy",color="r")
plt.plot(clean_data[shot_idx,:,rcv_idx],label="clean",color="k")
plt.legend()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

shot_idx = 0
rcv_idx = 0
dt = 0.003

# Select a single trace
trace_clean = clean_data[shot_idx, :, rcv_idx]
trace_noisy = noisy_data[shot_idx, :, rcv_idx]

# Number of samples and sampling interval
nt = trace_clean.shape[0]

# Frequency axis
freqs = np.fft.rfftfreq(nt, dt)

# FFT
fft_clean = np.fft.rfft(trace_clean)
fft_noisy = np.fft.rfft(trace_noisy)

# Amplitude spectrum
amp_clean = np.abs(fft_clean)
amp_noisy = np.abs(fft_noisy)

# Plot
plt.figure(figsize=(10,4))
plt.semilogy(freqs, amp_clean, label="Clean", color="k")
plt.semilogy(freqs, amp_noisy, label="Noisy", color="r", alpha=0.7)
plt.xlabel("Frequency [Hz]")
plt.ylabel("Amplitude")
plt.title(f"Frequency Spectrum of Trace shot={shot_idx}, rcv={rcv_idx}")
plt.legend()
plt.xlim(0, 50)  # zoom into relevant band
plt.show()


## Article Figure

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import gridspec
plt.rcParams['svg.fonttype'] = 'none'

def normalize(data):
    """Normalize data to [-1, 1]"""
    return ((data - data.min()) / (data.max() - data.min()) - 0.5) * 2

shot_idx = 20
rcv_idx = 50

resample_rate_t = 5
resample_rate_x = 1

# meshgrid for pcolormesh
t = np.arange(nt//resample_rate_t) * dt * resample_rate_t
r = np.arange(nx)
z_mesh, x_mesh = np.meshgrid(r, t)

# figure layout
fig = plt.figure(figsize=(12, 14))
gs = gridspec.GridSpec(3, 2, width_ratios=[1, 1], height_ratios=[3, 1, 1])
ax00 = fig.add_subplot(gs[0, 0])
ax01 = fig.add_subplot(gs[0, 1])
ax1  = fig.add_subplot(gs[1, :])
ax2  = fig.add_subplot(gs[2, :])

# colormesh: clean vs noisy
vmin, vmax = -0.005, 0.005
pcm0 = ax00.pcolormesh(z_mesh, x_mesh, clean_data[shot_idx,::resample_rate_t,::resample_rate_x], cmap="seismic", vmin=vmin, vmax=vmax)
pcm1 = ax01.pcolormesh(z_mesh, x_mesh, noisy_data[shot_idx,::resample_rate_t,::resample_rate_x], cmap="seismic", vmin=vmin, vmax=vmax)

# mark receiver
for ax, color in zip([ax00, ax01], ["k", "r"]):
    ax.scatter(rcv_idx, -0.1, marker="v", color=color, s=500, zorder=10)
    ax.invert_yaxis()
    ax.set_ylim(nt * dt, 0)
    ax.set_xlabel("Receiver Number", fontsize=14)
    ax.set_ylabel("Time (s)", fontsize=14)
    ax.tick_params(labelsize=14)

ax00.set_title("Clean Shot Gather", fontsize=15, weight="bold")
ax01.set_title("Noisy Shot Gather", fontsize=15, weight="bold")

# single trace comparison
ax1.plot(t, normalize(clean_data[shot_idx, ::resample_rate_t, rcv_idx]), color="k", lw=1.5, label="Clean Trace")
ax1.plot(t, normalize(noisy_data[shot_idx, ::resample_rate_t, rcv_idx]), color="r", lw=1.2, label="Noisy Trace")
ax1.set_ylabel("Normalized Amp.", fontsize=14)
ax1.set_xlabel("Time (s)", fontsize=14)
ax1.tick_params(labelsize=14)
ax1.legend(fontsize=12, loc="upper right")
ax1.grid(alpha=0.3)

# frequency spectrum
freq = np.fft.rfftfreq(nt, dt)
amp_clean = np.abs(np.fft.rfft(clean_data[shot_idx, :, rcv_idx]))
amp_noisy = np.abs(np.fft.rfft(noisy_data[shot_idx, :, rcv_idx]))

ax2.semilogy(freq, amp_clean, color="k", lw=1.5, label="Clean Spectrum")
ax2.semilogy(freq, amp_noisy, color="r", lw=1.2, label="Noisy Spectrum")
ax2.set_xlim(0, 50)
ax2.set_xlabel("Frequency (Hz)", fontsize=14)
ax2.set_ylabel("Amp.", fontsize=14)
ax2.tick_params(labelsize=14)
ax2.legend(fontsize=12, loc="upper right")
ax2.grid(alpha=0.3)

# layout
plt.tight_layout()

fig.text(0.030, 0.97, "(a)", fontsize=16, fontweight='bold', ha="center", va="center")
fig.text(0.520, 0.97, "(b)", fontsize=16, fontweight='bold', ha="center", va="center")
fig.text(0.030, 0.42, "(c)", fontsize=16, fontweight='bold', ha="center", va="center")
fig.text(0.030, 0.21, "(d)", fontsize=16, fontweight='bold', ha="center", va="center")


# plt.savefig("./Figures/FigureS1_RealStyle_Noise.png",dpi=300,bbox_inches="tight")
plt.savefig("./Figures_SVG/FigureS1_RealStyle_Noise.svg",bbox_inches="tight",format="svg")
plt.show()